In [2]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


## Get the sentiment and confidence score of a sentence

In [3]:
import gradio as gr
from transformers import pipeline
import pandas as pd
import numpy as np

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

model = pipeline(model = model_path, tokenizer = model_path)

def call_model(text):
  sentiment = model(text)[0]['label']
  score = model(text)[0]['score']
  return sentiment, score

demo = gr.Interface(fn=call_model,
                    inputs=["textbox"],
                    outputs=["textbox"])
demo.launch()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0bd58ba3a6de80e69c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


***

## Convert a csv review file into json

In [4]:
import json
from transformers import pipeline
import pandas as pd
import numpy as np

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
model = pipeline(model=model_path, tokenizer=model_path, truncation=True)

def call_model(text):
    result = model(text, truncation=True, max_length=512)[0]
    return result['label'], result['score']

def load_data(file,sample_size):
    df = pd.read_csv(file.name, encoding="utf-8")

    matching_cols = [col for col in df.columns if "text" in col.lower()]
    if matching_cols:
        review_col = matching_cols[0]
        review_df = df[review_col].dropna()
        reviews = (
            review_df[~review_df.str.contains("�")]
            .apply(lambda x: x.encode("utf-8", "ignore").decode("utf-8"))
            .reset_index(drop=True)
            .head(sample_size)
            .tolist()
        )
        return reviews
    print("No 'text' or 'review' column found in the DataFrame.")
    return []


def json_creation(file,sample_size):

  reviews = load_data(file,sample_size)
  if not reviews:
        return "No valid reviews found in the uploaded file."

  review_dict = {}
  for idx, review in enumerate(reviews):
      sentiment, confidence = call_model(review)
      review_dict[idx] = {"review": review, "sentiment": sentiment, "score": confidence}

  #Write Json
  json_path = "sentiment_results.json"
  with open(json_path, "w", encoding="utf-8") as f:
      json.dump(review_dict, f, ensure_ascii=False, indent=4)

  return json_path


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [5]:
import gradio as gr

demo = gr.Interface(fn=json_creation,
                    inputs=[
                        gr.File(label="Upload CSV File"),
                        gr.Slider(10, 30000, step=10, value=200, label="Sample Size")],
                    outputs=gr.File(label="Download Sentiment JSON"))
demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc184092caf62116c0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk import word_tokenize
from nltk.probability import FreqDist
import urllib.request
from matplotlib import pyplot as plt
from wordcloud import WordCloud
#nltk.download('punkt_tab')
import string
import gensim.parsing.preprocessing as gp
import spacy
load_en = spacy.load('en_core_web_sm')


def load_data(file):
    try:
        with open(file.name, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except Exception as e:
        return {"error": str(e)}

def get_by_sentiment(review_dict, sentiment):
  filtered_dict = {}
  for key, value in review_dict.items():
    if value['sentiment'] == sentiment:
      filtered_dict[key] = value
  return filtered_dict

##########################################################

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

def lemmatize_and_remove_stopwords(text):
    words = load_en(text)
    result = [lem_word.lemma_ for lem_word in words if not lem_word.is_stop]
    return result

def process_list(review_list):
  processed_list = [lemmatize_and_remove_stopwords(gp.strip_multiple_whitespaces(remove_punctuation(review)))for review in review_list]
  return processed_list

def process_reviews(rdict,sentiment):
  correct_sentiment_list = []
  other_sentiment_list = []
  for key, value in rdict.items():
    if value['sentiment'] == sentiment:
      correct_sentiment_list.append(value['review'])
    else:
      other_sentiment_list.append(value['review'])

  review_list = process_list(correct_sentiment_list)
  other_review_list = process_list(other_sentiment_list)
  return review_list, other_review_list


def get_frequency(review_list, other_review_list):
  fdist_list = [FreqDist(word) for word in review_list]
  antifdist_list = [FreqDist(word) for word in other_review_list]

  wfrequence = FreqDist()
  for fdist in fdist_list:
      wfrequence.update(fdist)
  frequence = wfrequence.most_common()

  for fdist in antifdist_list:
      wfrequence.update(fdist)
  anti_frequence = wfrequence.most_common()

  return frequence, anti_frequence


def filter_by_sentiment(sentiment_words, anti_sentiment_words):
  most_common_words = dict(sentiment_words)
  other_most_common_words = dict(anti_sentiment_words)

  word_importance_ratio = {
    word: (freq + 1) / (other_most_common_words.get(word, 0) + 1)
    for word, freq in most_common_words.items()
  }

  specific_words = {word: score for word, score in word_importance_ratio.items() if score > 0.5}

  specific_words_sorted = sorted(specific_words.items(), key=lambda x: x[1], reverse=True)

  return specific_words_sorted

def plot_wc(file, sentiment):

  data = load_data(file)
  #rdict = get_by_sentiment(data)
  rlist, orlist = process_reviews(data,sentiment)
  frequence, anti_frequence = get_frequency(rlist, orlist)
  most_common_words = filter_by_sentiment(frequence, anti_frequence)

  wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(dict(most_common_words))

  fig, ax = plt.subplots(figsize=(10, 5))
  ax.imshow(wordcloud, interpolation='bilinear')
  ax.axis('off')

  return fig

In [11]:
import gradio as gr

with gr.Blocks() as demo:
  gr.Markdown("""
    ### This webapp allows you to generate the word cloud for the most frequent words based on the desired sentiment for a review Json.

    ### 🔹 How to Use:
    1. **Upload a Json file** with the reviews.
    2. **Select a sentiment**: Positive, Negative or Neutral.
    4. A word cloud is generated.
    """)

  gr.Interface(fn=plot_wc,
                    inputs=[
                        gr.File(label="Upload JSON File"),
                        gr.Radio(['positive', 'neutral', 'negative'], label="Sentiment:")],
                    outputs=gr.Plot(label="wordcloud", format="png"))
demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e0f539c7c5c6d956e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
